# Setup Project Environment
1. Create neo_bank catalog, metadata schema
2. Create metadata tables: tables, table_parameters, table_watermarks, pipeline_runs, table_columns
3. insert values into metadata tables

In [0]:
CREATE CATALOG IF NOT EXISTS neo_bank;

CREATE SCHEMA IF NOT EXISTS neo_bank.metadata;


In [0]:
CREATE TABLE IF NOT EXISTS neo_bank.metadata.tables (
    table_id INT,
    table_name STRING,
    source_system STRING,      -- sql server/blob
    source_schema STRING,      -- dbo (null for blob)
    source_table STRING,       -- table name (null for blob)
    source_path STRING,        -- blob path (null for sqlserver)
    target_layer STRING,       -- silver/gold
    bronze_schema STRING,      -- bronze
    silver_schema STRING,      -- silver
    gold_schema STRING,        -- gold
    active_flag BOOLEAN,
    load_order INT,
    created_at TIMESTAMP
)
USING DELTA;


CREATE TABLE IF NOT EXISTS neo_bank.metadata.table_parameters (
    table_id INT,
    parameter_name STRING,  -- load_type / primary_key / watermark_column
    parameter_value STRING,
    created_at TIMESTAMP
)
USING DELTA;

CREATE TABLE IF NOT EXISTS neo_bank.metadata.table_watermarks (
    table_id INT,
    last_watermark_value STRING,
    last_updated_at TIMESTAMP,
    last_run_id BIGINT
)
USING DELTA
PARTITIONED BY (table_id);

CREATE TABLE IF NOT EXISTS neo_bank.metadata.pipeline_runs (
    run_id BIGINT,
    table_id INT,
    layer STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    number_of_records BIGINT,
    error_message STRING
)
USING DELTA 
PARTITIONED BY (table_id);

CREATE TABLE IF NOT EXISTS neo_bank.metadata.table_columns (
    table_id INT,
    target_schema STRING

)
USING DELTA;

In [0]:
INSERT INTO neo_bank.metadata.tables VALUES 
(1, 'accounts', 'neo_bank_db', 'banking', 'accounts', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 1, current_timestamp()),
(2, 'branches', 'neo_bank_db', 'banking', 'branches', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 2, current_timestamp()),
(3, 'customers', 'neo_bank_db', 'banking', 'customers', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 3, current_timestamp()),
(4, 'transactions', 'neo_bank_db', 'banking', 'transactions', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 4, current_timestamp()),
(5, 'credit_bureau_reports', 'volumes', NULL, NULL, '/Volumes/neo_bank/landing/files/credit_bureau_reports/', 'silver', 'bronze', 'silver', NULL, TRUE, 5, current_timestamp()),
(6, 'payment_gateway_logs', 'volumes', NULL, NULL, '/Volumes/neo_bank/landing/files/payment_gateway_logs/', 'silver', 'bronze', 'silver', NULL, TRUE, 6, current_timestamp())


In [0]:
INSERT INTO neo_bank.metadata.table_parameters VALUES
-- Accounts
(1,'load_type','MERGE',current_timestamp()),
(1,'primary_key','account_id',current_timestamp()),
(1,'watermark_column','updated_at',current_timestamp()),
-- Branches
(2,'load_type','FULL',current_timestamp()),
(2,'primary_key','branch_code',current_timestamp()),
-- Customers
(3, 'load_type','MERGE',current_timestamp()),
(3, 'primary_key','customer_id',current_timestamp()),
(3,'watermark_column','updated_at',current_timestamp()),
-- Transactions
(4, 'load_type', 'APPEND', current_timestamp()),
(4, 'primary_key','txn_id',current_timestamp()),
(4, 'watermark_column','txn_timestamp',current_timestamp()),
-- credit_bureau_reports
(5, 'load_type', 'MERGE', current_timestamp()),
(5, 'primary_key','customer_id',current_timestamp()),
(5,'watermark_column','bureau_pull_date',current_timestamp()),
-- payment_gateway_logs
(6, 'load_type','APPEND',current_timestamp()),
(6, 'primary_key', 'txn_id',current_timestamp()),
(6, 'watermark_column','processed_timestamp',current_timestamp())

In [0]:
--- since we have very few branches we just do full load for it instead of incremental load

INSERT INTO neo_bank.metadata.table_watermarks VALUES 
(1, '1900-01-01 00:00:00', current_timestamp(), NULL),
(3, '1900-01-01 00:00:00', current_timestamp(), NULL),
(4, '1900-01-01 00:00:00', current_timestamp(), NULL),
(5, '1900-01-01 00:00:00', current_timestamp(), NULL),
(6, '1900-01-01 00:00:00', current_timestamp(), NULL);



In [0]:
INSERT INTO neo_bank.metadata.table_columns VALUES
(1,NULL),
(2,NULL),
(3,NULL),
(4,NULL),
(5,'{"customer_id":"INT","credit_score":"INT","risk_grade":"STRING","external_active_loans":"INT","external_overdue_amount":"INT","bureau_pull_date":"DATE"}'),
(6,'{"txn_id":"INT","gateway_name":"STRING","gateway_status":"STRING","response_code":"INT","processing_time_ms":"INT","device_type":"STRING","geo_location":"STRING","processed_timestamp":"TIMESTAMP"}');